In [ ]:
%pip install -q "deepagents>=0.5.2" langchain langgraph langchain-openai langchain-anthropic

In [ ]:
from pathlib import Path

ROOT = Path("demo_workspace").resolve()
skill = ROOT / "skills" / "demo-research"
skill.mkdir(parents=True, exist_ok=True)
(skill / "SKILL.md").write_text(
"""---
name: demo-research
description: Research a topic and write findings. Use for research tasks.
---

# demo-research

1. write_todos: plan, research, write report
2. task → research-worker for the topic
3. write /workspace/report.md
"""
)

In [ ]:
import os

from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.agents.middleware import TodoListMiddleware
from langgraph.checkpoint.memory import MemorySaver

MODEL = (
    "openai:gpt-4.1-mini"
    if os.environ.get("OPENAI_API_KEY")
    else "anthropic:claude-sonnet-4-6"
)

backend = FilesystemBackend(root_dir=str(ROOT), virtual_mode=True)

research_worker = {
    "name": "research-worker",
    "description": "Researches one topic; returns a short summary.",
    "system_prompt": "Research only the assigned topic. Return 5-8 bullets.",
    "skills": ["/skills/"],
}

agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "Always write_todos. Always task(subagent_type='research-worker') for research. "
        "Then write /workspace/report.md."
    ),
    middleware=[TodoListMiddleware()],
    subagents=[research_worker],
    skills=["/skills/"],
    backend=backend,
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": "demo-1"}}
result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Use demo-research. Research why isolated subagents help context. "
                "Write /workspace/report.md."
            ),
        }]
    },
    config=config,
)

for m in result["messages"]:
    text = m.content
    if isinstance(text, list):
        text = " ".join(
            p.get("text", str(p)) if isinstance(p, dict) else str(p) for p in text
        )
    print(f"\n=== {m.type} ===\n{str(text)[:2000]}")

print("\n=== todos ===\n", result.get("todos"))
report = ROOT / "workspace" / "report.md"
print("\n=== report ===\n", report.read_text() if report.exists() else "(missing)")

In [ ]:
from langchain.tools import tool
from langgraph.types import Command, interrupt


@tool
def choose_skill(options: list[str]) -> str:
    """Pause so the user picks one skill name."""
    choice = interrupt({"type": "skill_choice", "options": options})
    if isinstance(choice, dict):
        return str(choice.get("skill") or choice.get("selected") or choice)
    return str(choice)


skill_worker = {
    "name": "skill-worker",
    "description": "Runs skill todos one by one after the user picks a skill.",
    "system_prompt": (
        "write_todos then complete them in order. "
        "Write /workspace/skill-result.md when done."
    ),
    "skills": ["/skills/"],
}

design = create_deep_agent(
    model=MODEL,
    tools=[choose_skill],
    middleware=[TodoListMiddleware()],
    subagents=[skill_worker],
    skills=["/skills/"],
    backend=backend,
    checkpointer=MemorySaver(),
    system_prompt=(
        "Call choose_skill(options=['demo-research']). "
        "After pick, task once with skill-worker."
    ),
)

config2 = {"configurable": {"thread_id": "demo-skill-pick"}}
paused = design.invoke(
    {"messages": [{"role": "user", "content": "Start."}]},
    config=config2,
)
print("interrupt:", paused.get("__interrupt__"))

resumed = design.invoke(
    Command(resume={"skill": "demo-research"}),
    config=config2,
)
print("final:", resumed["messages"][-1].content)
print("todos:", resumed.get("todos"))
out = ROOT / "workspace" / "skill-result.md"
print("result file:", out.read_text() if out.exists() else "(missing)")